# English Premier League 2023/24 — Exploratory Data Analysis
**By:** Youssef Moussa
**Date:** 21/3/2026
**Dataset:** EPL 2023/24 Match Stats (one row = one team's match performance)

---

### Goals of This Analysis
- Understand the structure and quality of the dataset
- Clean and prepare data for analysis
- Explore distributions of key performance metrics
- Discover patterns across teams, venues, and match results

## Stage 1 — Setup & First Look

### 1.1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1.2 Load Dataset

In [ ]:
df = pd.read_csv('matches.csv')
df.sample(3)

**Observation:**
- Each row represents a single team's performance in one match
- For example, a Man City vs Arsenal match appears as **two separate rows** — one per team

### 1.3 Dataset Shape & Structure

In [ ]:
print(f"Shape: {df.shape}")
df.info()

**Observation:**
- 760 rows × 28 columns
- `Notes` column has 0 non-null values — entirely empty
- `Date` is stored as string — needs conversion to datetime
- `Attendance` is float64 — suggests missing values (NaN forces float in pandas)

### 1.4 Statistical Summary

In [ ]:
df.describe()

**Observation:**
- Quick scan shows `Poss` mean = 50.0, which makes perfect sense — possession always sums to 100% between both teams in a match

### 1.5 Missing Values Check

In [ ]:
df.isna().sum()

**Observation:**
- `Notes`: 760 nulls (100% empty) → will be dropped
- `Attendance`: has some nulls → will be filled with team median
- All other columns: complete, no missing values

---
## Stage 2 — Data Cleaning

### 2.1 Drop Useless Columns
Dropping columns with zero analytical value:
- `Unnamed: 0` — leftover index from the CSV export
- `Notes` — entirely null (760/760 missing values)
- `Match Report` — every value is the string `'Match Report'` (a link label, not data)
- `Comp` — only one unique value: `'Premier League'` (no variance)
- `Season` — only one unique value: `2024` (no variance)

In [ ]:
# Verify zero-variance columns before dropping
for col in ['Match Report', 'Comp', 'Season']:
    print(f"{col}: {df[col].nunique()} unique value(s)")

cols_to_drop = ['Unnamed: 0', 'Notes', 'Match Report', 'Comp', 'Season']
df.drop(columns=cols_to_drop, inplace=True)

print(f"Remaining columns: {df.shape[1]}")
print(df.columns.tolist())

**Observation:**
- Dropped 5 columns
- Remaining 23 columns are all analytically useful

### 2.2 Fix Date Column dtype

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
print(f"Date dtype: {df['Date'].dtype}")

**Observation:**
- Date is now a proper datetime object
- We can now extract `month`, `weekday`, and calculate time differences between matches

### 2.3 Add Goal Difference Column (GD)

In [ ]:
df['GD'] = df['GF'] - df['GA']
df[['GF', 'GA', 'GD', 'Result']].head()

**Observation:**
- `GD` captures match dominance beyond just W/D/L
- Winning 5-0 and 1-0 are both wins, but GD tells the real story

### 2.4 Verify Possession Range (0–100%)

In [ ]:
impossible_poss = df[(df['Poss'] > 100) | (df['Poss'] < 0)]
print(f"Impossible Poss values: {len(impossible_poss)}")

**Observation:**
- No impossible values found
- `Poss` is clean and bounded within 0–100%

### 2.5 Check for Outliers in Possession (IQR Method)

In [ ]:
Q1 = df['Poss'].quantile(0.25)
Q3 = df['Poss'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['Poss'] < lower) | (df['Poss'] > upper)]
print(f"Outliers found: {len(outliers)}")
print(f"IQR bounds: [{lower:.1f}, {upper:.1f}]")
print(df['Poss'].describe())

**Observation:**
- No outliers detected
- Distribution is perfectly symmetrical around 50 — expected, since both teams' possession always sums to 100% in any match

### 2.6 Check for Duplicate Rows

In [ ]:
dupl_mask = df.duplicated()
print(f"Duplicate rows: {dupl_mask.sum()}")

**Observation:**
- No duplicate rows found
- Dataset is clean

### 2.7 Add Rest Days Column

In [ ]:
# Sort by team and date to calculate consecutive match gaps
df = df.sort_values(['Team', 'Date']).reset_index(drop=True)

# Calculate days between consecutive matches per team
df['Rest_Days'] = df.groupby('Team')['Date'].diff().dt.days

# Fill NaN with 0 (first match per team has no prior match)
df['Rest_Days'] = df['Rest_Days'].fillna(0).astype('Int64')

print(df['Rest_Days'].describe())

**Observation:**
- Mean rest ≈ 7.3 days — consistent with the weekly EPL schedule
- Max = 23 days — likely due to an international break
- 20 zeros represent the first match for each of the 20 teams (fill value, not real rest)

### 2.8 Handle Attendance Missing Values

In [ ]:
print(f"Attendance nulls before fill: {df['Attendance'].isnull().sum()}")

# Fill with each team's median attendance (smarter than global mean)
df['Attendance'] = df.groupby('Team')['Attendance'] \
                     .transform(lambda x: x.fillna(x.median()))

print(f"Attendance nulls after fill: {df['Attendance'].isnull().sum()}")

# Convert to nullable integer
df['Attendance'] = df['Attendance'].astype('Int64')

**Observation:**
- Used team median instead of global mean to preserve each stadium's realistic capacity
- A small ground like Luton's should not be filled with the league-wide average

### 2.9 Final Clean Dataset Overview

In [ ]:
print(f"Final shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nRemaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

**Observation:**
- Dataset is clean and ready for analysis
- 760 rows × 25 columns with no remaining null values

---
## Stage 3 — Univariate Analysis

### 3.1 Distribution of Goals Scored (GF)
Analyzing the most common number of goals scored per match, the direction of skew, and any surprisingly high values.

In [ ]:
most_three_scored_goals = df['GF'].value_counts(ascending=False).head(3)
print(f"Most three common goals scored:\n{most_three_scored_goals}")
print(f"\nSkewness: {df['GF'].skew():.2f}")

df['GF'].value_counts()

In [ ]:
df['GF'].plot(
    kind='hist',
    bins=10,
    title='Distribution of Goals Scored per Match',
    ylabel='Frequency',
    xlabel='Goals Scored',
    edgecolor='black'
)
plt.axvline(x=df['GF'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(x=df['GF'].median(), color='green', linestyle='-', label='Median')
plt.legend()
plt.show()

**Observation:**
- The three most common goals scored are 1, 2, and 0
- The distribution is right-skewed — rare high-scoring matches pull the mean above the median
- Surprising high values: 8 goals (1 match), 6 goals (6 matches), 5 goals (19 matches)

### 3.2 Match Result Distribution (Result)
Analyzing which result — Win, Draw, or Loss — appears most often across all 760 team-match performances.

In [ ]:
most_common_result = df['Result'].value_counts(ascending=False)

most_common_result.plot(
    kind='bar',
    xlabel='Result',
    ylabel='Occurrences',
    title='Match Result Distribution',
)
plt.xticks(rotation=0)
plt.show()

print(most_common_result)

**Observation:**
- There are 298 wins and 298 losses — equal by definition, since every win for one team is a loss for the opponent
- There are 164 draws — the least common outcome
- Decisive results dominate the EPL 2023/24 season

### 3.3 Possession Distribution (Poss)
Analyzing the distribution of possession percentage across all matches.

In [ ]:
df['Poss'].plot(
    kind='hist',
    bins=10,
    xlabel='Possession Percentage',
    ylabel='Frequency',
    title='Possession Percentage per Match',
    edgecolor='black'
)
plt.axvline(x=df['Poss'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(x=df['Poss'].median(), color='green', linestyle='-', label='Median')
plt.xticks(np.arange(20, 85, 5))
plt.legend()
plt.show()

# Verify that mean and median overlap
print(f"Poss Mean: {df['Poss'].mean():.2f}, Poss Median: {df['Poss'].median():.2f}")

**Observation:**
- Mean and Median are both exactly 50.0 — confirming perfect symmetry
- This makes mathematical sense: both teams' possession always sums to 100%
- Possession is fairly balanced across most matches — no single team consistently dominates the ball, reflecting the competitive nature of the EPL

### 3.4 Shots on Target Distribution (SoT)
Analyzing the typical range of shots on target per match and identifying any surprisingly high values.

**Pre-plot prediction:** Expected most common values to be around 4, 3, and 5 — similar shape to GF (right-skewed).

In [ ]:
sot = df['SoT']

print(f"Top 3 most common SoT values:\n{sot.value_counts(ascending=False).head(3)}")
print(f"\nSkewness: {sot.skew():.2f}")

sot.value_counts(ascending=True)

In [ ]:
sot.plot(
    kind='hist',
    title='Shots on Target per Match',
    xlabel='Shots on Target',
    ylabel='Frequency',
    edgecolor='black'
)
plt.axvline(x=sot.mean(), linestyle='-', label='Mean', color='red')
plt.axvline(x=sot.median(), linestyle='--', label='Median', color='green')
plt.legend()
plt.show()

**Observation:**
- Distribution is right-skewed (skewness = 0.84) — most teams register 3–5 shots on target, but a few high-performing matches pull the mean above the median
- Most common value is 4, with a frequency of approximately 250
- Surprisingly high values exist in the 10–14 range, though with frequency below 50
- Some matches recorded 0 shots on target — indicating very poor attacking performance

### 3.5 Rest Days Distribution (Rest_Days)
Analyzing the distribution of rest days between consecutive matches per team.

**Pre-plot prediction:** Expected most common rest window to be around 7, 8, and 3 days — consistent with the weekly EPL schedule and midweek fixtures.

In [ ]:
# Exclude the 20 fill zeros (first match per team — no prior match to diff against)
rest = df[df['Rest_Days'] != 0]['Rest_Days']

print(f"Skewness: {rest.skew():.2f}")
rest.value_counts().sort_index()

In [ ]:
rest.plot(
    kind='hist',
    xlabel='Rest Days',
    ylabel='Frequency',
    title='Rest Days Between Matches per Team',
    edgecolor='black'
)
plt.axvline(rest.mean(), label='Mean', color='red', linestyle='-')
plt.axvline(rest.median(), label='Median', color='green', linestyle='--')
plt.legend()
plt.show()

**Observation:**
- Most common rest window is 7–9 days, consistent with the weekly EPL schedule
- The 3–5 day bin occurs approximately 160 times — teams playing twice a week during congested fixture periods
- Surprising values of 20+ days are likely due to international breaks
- Right-skewed (skewness = 1.46) — long breaks pull the mean above the median

### 3.6 Summary Statistics Table

In [ ]:
def summary_stats(column):
    return [
        round(column.mean(), 2),
        round(column.median(), 2),
        round(column.std(), 2),
        round(column.skew(), 2)
    ]

summary_df = {
    'GF':        summary_stats(df['GF']),
    'GD':        summary_stats(df['GD']),
    'Poss':      summary_stats(df['Poss']),
    'SoT':       summary_stats(df['SoT']),
    'Rest_Days': summary_stats(rest),   # zeros excluded
}

summary_df = pd.DataFrame(summary_df, index=['Mean', 'Median', 'Std', 'Skewness'])
summary_df